# HMM (Simplified)
3 hidden intent states: Low, Medium, High intent.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

demo = pd.read_csv(Path('../data/newdata.csv'))
feature_cols = [
    'session_length_s', 'avg_time_between_events_s', 'cart_to_view_ratio',
    'number_of_cart_additions', 'number_of_purchases', 'time_to_cart_s',
    'time_to_purchase_s', 'engagement_score', 'conversion_intent_score'
]

X = demo[feature_cols].values.astype(float)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

hmm = GaussianHMM(n_components=3, covariance_type='diag', n_iter=200, random_state=42)
hmm.fit(Xs)

hidden = hmm.predict(Xs)
demo['hidden_state_id'] = hidden
demo[['session_id', 'hidden_state_id']].head(10)

,session_id,hidden_state_id
0,1,0
1,2,2
2,3,1
3,4,0
4,5,1
5,6,2
6,7,1
7,8,0
8,9,2
9,10,1


In [3]:
intent_by_state = demo.groupby('hidden_state_id')['conversion_intent_score'].mean().sort_values()
order = list(intent_by_state.index)
name_map = {order[0]: 'Low Intent', order[1]: 'Medium Intent', order[2]: 'High Intent'}
demo['hidden_state_name'] = demo['hidden_state_id'].map(name_map)

hidden_transitions = pd.DataFrame(
    hmm.transmat_,
    index=[name_map[i] for i in range(3)],
    columns=[name_map[i] for i in range(3)]
)

print('Hidden transition matrix:')
print(hidden_transitions.round(3))

print('\nMean feature values by hidden state:')
display(demo.groupby('hidden_state_name')[feature_cols].mean().round(3))

plt.figure(figsize=(5, 4))
sns.heatmap(hidden_transitions, annot=True, fmt='.2f', cmap='Blues')
plt.title('HMM Hidden-State Transition Heatmap')
plt.tight_layout()
plt.show()

demo[['session_id', 'hidden_state_name'] + feature_cols].head(20)

Hidden transition matrix:
               Low Intent  High Intent  Medium Intent
Low Intent            0.0          0.4            0.6
High Intent           0.5          0.0            0.5
Medium Intent         0.0          1.0            0.0

Mean feature values by hidden state:


,session_length_s,avg_time_between_events_s,cart_to_view_ratio,number_of_cart_additions,number_of_purchases,time_to_cart_s,time_to_purchase_s,engagement_score,conversion_intent_score
hidden_state_name,,,,,,,,,
High Intent,791.250,36.500,0.412,2.500,1.0,78.125,376.875,7.975,8.250
Low Intent,300.000,54.000,0.054,0.000,0.0,-1.000,-1.000,1.560,-0.040
Medium Intent,484.286,43.714,0.226,1.143,0.0,128.571,-1.000,3.600,2.729


,session_id,hidden_state_name,session_length_s,avg_time_between_events_s,cart_to_view_ratio,number_of_cart_additions,number_of_purchases,time_to_cart_s,time_to_purchase_s,engagement_score,conversion_intent_score
0,1,Low Intent,320,45,0.10,0,0,-1,-1,1.8,0.2
1,2,Medium Intent,540,38,0.25,1,0,120,-1,3.6,2.8
2,3,High Intent,780,42,0.33,2,1,95,420,6.8,7.2
3,4,Low Intent,260,60,0.00,0,0,-1,-1,1.1,-0.3
4,5,High Intent,640,36,0.40,2,1,80,390,7.4,7.8
5,6,Medium Intent,410,48,0.20,1,0,140,-1,3.2,2.2
6,7,High Intent,900,34,0.50,3,1,70,360,8.9,9.1
7,8,Low Intent,300,55,0.00,0,0,-1,-1,1.4,-0.5
8,9,Medium Intent,520,40,0.28,1,0,115,-1,3.9,3.1
9,10,High Intent,850,33,0.45,3,1,65,340,9.2,9.5
